## If a key leaks anyway

1. **Revoke it first, investigate later.** Go to [openrouter.ai/keys](https://openrouter.ai/keys) and delete the key immediately. Every minute it stays active is a minute a bot can use it.
2. **Don't rely on removing it from git history.** Force-pushing a "fix" that deletes the key from an old commit does not undo a leak — if the repo was ever public, the key may already be scraped and stored elsewhere, independent of your repository's current state. Revocation, not cleanup, is what actually stops the exposure.
3. **Generate a new key and update everywhere it's used** — any local `.env` files, anywhere else you referenced it.
4. **Check for damage.** Look at [openrouter.ai/activity](https://openrouter.ai/activity) for requests you didn't make, and address any unexpected charges with OpenRouter directly.

A leaked key is billed from the first unauthorized request. That's exactly why revoking fast matters, and why the disclaimer at the top of this notebook means what it says: you're responsible for watching your own key, this notebook can only tell you how.

## Why this matters

An API key is a password. Anyone who has it can spend against your account, and on GitHub, "anyone" includes automated bots built specifically to find keys the moment they go public.

GitHub scans every public push for text that matches the shape of a known key format, and for over a hundred providers — including Google Cloud, OpenAI, and Anthropic — it reports a match straight to that provider, who revokes it automatically. That's the good outcome. The bad outcome is what happens with everything else: independent scanners run the same search and aren't trying to help you. In one widely cited case, a developer's leaked AWS credentials were found and used by attackers in about six minutes; in another, bots spun up 500+ cloud instances on a leaked key within twenty minutes, running up a $72,000 bill before the owner noticed. Nothing about this course's usage is immune to the same pattern — a leaked key just gets you cut off, or billed, faster than you can react.

## Setup guidelines

1. **Get an OpenRouter key and put some credit on the account.** See Week 2's setup guide. The lightweight models this course uses cost fractions of a cent per request, but there's no free quota to fall back on, so keep an eye on usage from the start.
2. **Store it in a `.env` file, never in code.** The file lives in the project folder and looks like this:
   ```
   OPENROUTER_API_KEY=sk-or-v1-your-key-here
   ```
   Load it with:
   ```python
   from dotenv import load_dotenv
   import os

   load_dotenv()
   api_key = os.getenv("OPENROUTER_API_KEY")
   ```
3. **Confirm `.env` is in `.gitignore`.** This is what actually keeps the key out of git, not the `.env` file by itself. If you created your repository the way Week 2 describes (checking "Add .gitignore," choosing "Python"), this is already handled. The cell below checks either way.
4. **Set a credit limit** on [openrouter.ai/settings/credits](https://openrouter.ai/settings/credits). It's the safety net for when every other guideline here fails.

In [ ]:
from pathlib import Path

def check_gitignore(text: str) -> bool:
    """Check whether a .gitignore file's contents would exclude a .env file."""
    patterns = [line.strip() for line in text.splitlines() if line.strip() and not line.startswith("#")]
    covers_env = any(p in (".env", "*.env", ".env*") for p in patterns)
    if covers_env:
        print("Covered: .env will not be tracked by git.")
    else:
        print("Not covered: add a line containing exactly `.env` to your .gitignore.")
    return covers_env

gitignore_path = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / ".gitignore").exists():
        gitignore_path = candidate / ".gitignore"
        break

if gitignore_path is not None:
    print("Checking", gitignore_path)
    check_gitignore(gitignore_path.read_text(encoding="utf-8"))
else:
    print("No local .gitignore found here -- checking a sample instead:\n")
    sample_gitignore = "__pycache__/\n*.pyc\n.env\n.DS_Store\n"
    print(sample_gitignore)
    check_gitignore(sample_gitignore)

## Protection guidelines: three ways a key gets out, even with `.env` in place

**1. Never type it directly into a code cell.** `api_key = "sk-or-v1-..."` works exactly as well as a `.env` file, right up until you save the notebook to GitHub — at which point the key is sitting in plain text in a public file, permanently, in your commit history, even if you delete the line in a later commit. *Do instead*: load it from `.env`, always.

**2. Never print it in full.** A notebook file saves its cell *outputs* along with the code, not just the code you wrote — so if a cell prints the whole key, that value gets written into the `.ipynb` file itself and pushed with it. This is true even if the key was loaded safely from `.env`. *Do instead*: print a truncated preview, like Week 2's setup notebook does — `api_key[:10] + "..."` — any time you print something that came from an `.env` file.

**3. Never paste it into an AI chat unredacted.** Debugging often means pasting an error message, a config file, or a `.env` into a chatbot to ask what's wrong, and it's easy for a real key to be sitting inside that paste without you noticing. A key pasted into a chat never touches GitHub, so none of the scanning described above catches it either. *Do instead*: scan a paste once for anything starting with a provider's key prefix (`sk-or-v1-`, `sk-`, `AIza`, and similar) and replace it with `[REDACTED]` before you hit send.

## Try it: what a scanner actually looks for

GitHub's scanner (and every bot copying its approach) isn't reading your code for meaning — it's pattern-matching on shape. An OpenRouter key starts with `sk-or-v1-` followed by a long run of letters and digits; other providers have their own fixed prefixes. Below is a tiny version of that same idea: a regular expression (a pattern for matching text) that flags anything shaped like a key, run over a few example lines from the guidelines above.

The "keys" below are made up and will not work against any real API — they exist only to show the shape a scanner is looking for.

In [ ]:
import re

# A simplified version of the pattern a scanner uses for OpenRouter keys.
# Real scanners also validate the key against the provider before alerting anyone --
# this version just checks the shape, which is enough to make the point.
OPENROUTER_KEY_PATTERN = re.compile(r"sk-or-v1-[0-9A-Za-z]{20,}")

# Fake examples only -- none of these are real, working keys.
sample_lines = [
    'client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))',  # safe
    'api_key = "sk-or-v1-FAKE00000000000000000000000000000000"',                                         # unsafe
    'print("Key loaded. Starts with:", api_key[:10] + "...")',                                           # safe
    'print("Key loaded:", api_key)',                                                                     # unsafe
    'response = client.chat.completions.create(model=MODEL_NAME, messages=messages)',                    # safe
]

for line in sample_lines:
    match = OPENROUTER_KEY_PATTERN.search(line)
    flag = "FLAGGED -- looks like a real key" if match else "clear"
    print(f"{flag:30s} | {line}")

## If a key leaks anyway

1. **Revoke it first, investigate later.** Go to [openrouter.ai/keys](https://openrouter.ai/keys) and delete the key immediately. Every minute it stays active is a minute a bot can use it.
2. **Don't rely on removing it from git history.** Force-pushing a "fix" that deletes the key from an old commit does not undo a leak — if the repo was ever public, the key may already be scraped and stored elsewhere, independent of your repository's current state. Revocation, not cleanup, is what actually stops the exposure.
3. **Generate a new key and update everywhere it's used** — any local `.env` files, anywhere else you referenced it.
4. **Check for damage.** Look at [openrouter.ai/activity](https://openrouter.ai/activity) for requests you didn't make, and address any unexpected charges with OpenRouter directly.

A leaked key is billed from the first unauthorized request. That's exactly why revoking fast matters, and why the disclaimer at the top of this notebook means what it says: you're responsible for watching your own key, this notebook can only tell you how.

## Protection guidelines: three ways a key gets out, even with `.env` in place

**1. Never type it directly into a code cell.** `api_key = "sk-or-v1-..."` works exactly as well as a `.env` file, right up until you save the notebook to GitHub — at which point the key is sitting in plain text in a public file, permanently, in your commit history, even if you delete the line in a later commit. *Do instead*: load it from `.env`, always.

**2. Never print it in full.** A notebook file saves its cell *outputs* along with the code, not just the code you wrote — so if a cell prints the whole key, that value gets written into the `.ipynb` file itself and pushed with it. This is true even if the key was loaded safely from `.env`. *Do instead*: print a truncated preview, like Week 2's setup notebook does — `api_key[:10] + "..."` — any time you print something that came from an `.env` file.

**3. Never paste it into an AI chat unredacted.** Debugging often means pasting an error message, a config file, or a `.env` into a chatbot to ask what's wrong, and it's easy for a real key to be sitting inside that paste without you noticing. A key pasted into a chat never touches GitHub, so none of the scanning described above catches it either. *Do instead*: scan a paste once for anything starting with a provider's key prefix (`sk-or-v1-`, `sk-`, `AIza`, and similar) and replace it with `[REDACTED]` before you hit send.